In [1]:
import sys, re
sys.path.append('..')  # Adjust the path as per your directory structure

from scripts.constants import *
from scripts.logging_config import *
from scripts.utils import *

import pandas as pd
import geopandas as gpd
import numpy as np
import rioxarray as rxr
from rioxarray.merge import merge_arrays
import rasterio as rio
from rasterstats import zonal_stats

In [2]:
geo_level = 'LAD22CD'
geo_code = 'E07000008'

In [3]:
 # IN paths
imd_lsoa_bua_boundaries_path = VECTOR_OUT_DIR / "IMD" / "English_IMD_2019_BUA_filtered_boundaries.geojson"
vom_dir = RASTER_IN_DIR / "Defra" / "VOM"
vom_lad_dir = vom_dir / "LADs"
vom_unzipped_dir = vom_dir / "unzipped_tiles"
vom_zipped_dir = vom_dir / "zipped_tiles"

tif_paths = list(vom_unzipped_dir.rglob("*.tif"))
tif_paths_lst = [[path.parent.name, extract_grid_reference(path.name), classify_vom_type(path.name), str(path)] for path in tif_paths]

tif_paths_df = pd.DataFrame(tif_paths_lst, columns=['year', 'TILE_NAME', 'file_type', 'path']).sort_values(['TILE_NAME', 'year']).reset_index(drop=True)

In [4]:
def generate_tiles_paths(geo_level: str, geo_code: str, imd_lsoa_bua_gdf: gpd.GeoDataFrame) -> tuple:
    """
    Generates tile paths and associated dataframes for a given geographical level and code.
    Parameters:
        geo_level (str): The geographical level (e.g., LSOA, BUA).
        geo_code (str): The geographical code corresponding to the geo_level.
        imd_lsoa_bua_gdf (gpd.GeoDataFrame): A GeoDataFrame containing geographical data.
    Returns:
        tuple: A tuple containing the following elements:
            - subgeo_filt_gdf (gpd.GeoDataFrame): A filtered GeoDataFrame for the specified geo_level and geo_code.
            - geo_boundary_gdf (gpd.GeoDataFrame): A dissolved GeoDataFrame containing the boundary geometry for the specified geo_level.
            - geo_selected_vom_tiles_df (pd.DataFrame): A DataFrame containing selected VOM tiles for the specified geo_code.
            - tif_paths_df (pd.DataFrame): A DataFrame containing paths to .tif files, along with their year, tile name, and file type.
    """

    logging.debug("Generating tile paths")

    subgeo_filt_gdf = imd_lsoa_bua_gdf.copy()[imd_lsoa_bua_gdf[geo_level].isin([geo_code])].reset_index(drop=True)

    geo_vom_tiles_path = vom_lad_dir / f"VOM_tiles_{geo_code}.csv"
    geo_vom_tiles_df = pd.read_csv(geo_vom_tiles_path).sort_values(['TILE_NAME', 'year'], ascending=[True, False])
    geo_selected_vom_tiles_df = geo_vom_tiles_df.groupby(['TILE_NAME']).first().reset_index()
    
    return subgeo_filt_gdf, geo_vom_tiles_df, geo_selected_vom_tiles_df

In [5]:
imd_lsoa_bua_gdf = gpd.read_file(imd_lsoa_bua_boundaries_path).sort_values(by=geo_level)
geo_level_codes = imd_lsoa_bua_gdf[geo_level].unique()

In [6]:
from tqdm import tqdm
geo_level_codes = imd_lsoa_bua_gdf[geo_level].unique()
chm_tiles_dict = {}
geo_code_errors = {}
for geo_code in tqdm(geo_level_codes):
    try:
        _, geo_vom_tiles_df, geo_selected_vom_tiles_df = generate_tiles_paths(geo_level, geo_code, imd_lsoa_bua_gdf)
        selected_chm_path_lst = select_chm_files(geo_vom_tiles_df, geo_selected_vom_tiles_df, tif_paths_df)

        chm_tiles_dict[geo_code] = [str(path) for path in selected_chm_path_lst]
    except Exception as e:
        geo_code_errors[geo_code] = type(e).__name__

100%|██████████| 309/309 [00:24<00:00, 12.69it/s]


In [8]:
longest_list_key = max(chm_tiles_dict, key=lambda k: len(chm_tiles_dict[k]))
longest_list = chm_tiles_dict[longest_list_key]
print(f"The longest list is for geo_code {longest_list_key} with {len(longest_list)} items.")

The longest list is for geo_code E06000057 with 225 items.


In [11]:
imd_lsoa_bua_gdf[imd_lsoa_bua_gdf['LAD22CD'] == 'E06000057']

,LSOA11CD,LSOA11NM,LSOA21CD,LSOA21NM,LSOA21NMW,LAD22CD,LAD22NM,LAD22NMW,BUA22CD,BUA22NMW,BUA22NMG,RGN22NMW,BUA22NM,RGN22CD,RGN22NM,geometry
30050,E01027504,Northumberland 039C,E01027504,Northumberland 043C,None,E06000057,Northumberland,None,None,None,None,None,None,None,None,"POLYGON ((397572.406 568256.125, 397678.813 56..."
30046,E01027500,Northumberland 038D,E01027500,Northumberland 038D,None,E06000057,Northumberland,None,E63000098,None,None,None,Prudhoe,E12000001,North East,"POLYGON ((409628.094 563008, 409819.905 562969..."
30048,E01027502,Northumberland 038F,E01035636,Northumberland 038G,None,E06000057,Northumberland,None,E63000098,None,None,None,Prudhoe,E12000001,North East,"POLYGON ((408846.567 562672.255, 408678.454 56..."
30047,E01027501,Northumberland 038E,E01035636,Northumberland 038G,None,E06000057,Northumberland,None,E63000098,None,None,None,Prudhoe,E12000001,North East,"POLYGON ((409032.119 562824.323, 409105.159 56..."
30051,E01027505,Northumberland 040E,E01027505,Northumberland 040E,None,E06000057,Northumberland,None,None,None,None,None,None,None,None,"POLYGON ((395335.595 561875.312, 395259.687 56..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36481,E01033715,Northumberland 005D,E01033715,Northumberland 005D,None,E06000057,Northumberland,None,E63000015,None,None,None,Alnwick,E12000001,North East,"POLYGON ((418778.706 613276.59, 419068.594 613..."
36482,E01033716,Northumberland 013F,E01033716,Northumberland 013F,None,E06000057,Northumberland,None,E63000040,None,None,None,Ashington (Northumberland),E12000001,North East,"POLYGON ((429968.5 587160.812, 430013.188 5869..."
36483,E01033717,Northumberland 005E,E01033717,Northumberland 005E,None,E06000057,Northumberland,None,None,None,None,None,None,None,None,"POLYGON ((418550.031 613191.269, 418564.318 61..."
29908,E01027357,Northumberland 004B,E01027357,Northumberland 004B,None,E06000057,Northumberland,None,E63000015,None,None,None,Alnwick,E12000001,North East,"POLYGON ((420335.406 613440.375, 420399.5 6132..."


In [26]:
import json

chm_tiles_dict
# Define the path to save the JSON file
json_output_path = vom_lad_dir / "LAD_CHM_tiles_paths.json"

# Dump the dictionary to a JSON file
with open(json_output_path, 'w') as json_file:
    json.dump(chm_tiles_dict, json_file)